In [128]:
import pandas as pd
import numpy as np

df=pd.read_json('D:/FM/candle_maker - Copy/artifacts/candle_data - 29-04-2026.json')
df['date']="29-04-2026"
df_2=pd.read_json('D:/FM/candle_maker - Copy/artifacts/candle_data - 30-04-2026.json')
df_2['date']="30-04-2026"
df = pd.concat([df, df_2], ignore_index=True)


In [129]:
df=df.drop(columns=['limit_orders'])

In [130]:
# Nested dicts (ohlcv, aggression) → flat columns: ohlcv_open, aggression_score, …
ohlc_flat = pd.json_normalize(df["ohlcv"]).add_prefix("ohlcv_")
agg_flat = pd.json_normalize(df["aggression"]).add_prefix("aggression_")

df = pd.concat(
    [df.drop(columns=["ohlcv", "aggression"], errors="ignore"), ohlc_flat, agg_flat],
    axis=1,
)


In [131]:
df['buy_v_pct']=round((df['aggression_buy_volume']/df['ohlcv_volume'])*100,2)
df['sell_v_pct']=round((df['aggression_sell_volume']/df['ohlcv_volume'])*100,2)

In [132]:
df

,symbol,candle_open,candle_close,timeframe,date,ohlcv_open,ohlcv_high,ohlcv_low,ohlcv_close,ohlcv_volume,aggression_score,aggression_buy_volume,aggression_sell_volume,buy_v_pct,sell_v_pct
0,NSE:ASIANPAINT-EQ,09:15:00,09:19:59,5min,29-04-2026,2449.1,2454.4,2439.3,2452.9,15106,-0.3739,4729,10377,31.31,68.69
1,NSE:HEROMOTOCO-EQ,09:15:00,09:19:59,5min,29-04-2026,5155.0,5170.0,5152.0,5158.5,16525,-0.0532,7823,8702,47.34,52.66
2,NSE:ASIANPAINT-EQ,09:20:00,09:24:59,5min,29-04-2026,2451.8,2455.8,2445.2,2449.6,10356,0.1688,6052,4304,58.44,41.56
3,NSE:HEROMOTOCO-EQ,09:20:00,09:24:59,5min,29-04-2026,5161.0,5175.0,5148.5,5160.5,19481,0.0278,10011,9470,51.39,48.61
4,NSE:ASIANPAINT-EQ,09:25:00,09:29:59,5min,29-04-2026,2450.9,2457.5,2450.0,2452.5,18535,-0.3498,6026,12509,32.51,67.49
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194,NSE:ASIANPAINT-EQ,14:15:00,14:19:59,5min,30-04-2026,2436.4,2439.4,2431.3,2431.8,9639,0.2962,6247,3392,64.81,35.19
195,NSE:ASIANPAINT-EQ,14:20:00,14:24:59,5min,30-04-2026,2431.9,2433.9,2430.0,2433.1,4017,0.1212,2252,1765,56.06,43.94
196,NSE:ASIANPAINT-EQ,14:25:00,14:29:59,5min,30-04-2026,2433.1,2434.0,2430.7,2434.0,9437,0.2578,5935,3502,62.89,37.11
197,NSE:ASIANPAINT-EQ,14:30:00,14:34:59,5min,30-04-2026,2434.1,2439.8,2432.8,2439.0,5376,0.4788,3975,1401,73.94,26.06


In [133]:
df['vol_filter'] = (
    df.groupby(['symbol', 'date'])['ohlcv_volume']
    .shift(1)
    .pipe(lambda prev: (df['ohlcv_volume'] > prev).astype(int))
)

In [134]:
df['domination'] = 'sell'
df.loc[df['buy_v_pct'] > df['sell_v_pct'], 'domination'] = 'buy'

In [135]:
df['buy_vol_filter'] = (
    df.groupby(['symbol', 'date'])['buy_v_pct']
    .shift(1)
    .pipe(lambda prev: (df['buy_v_pct'] > prev).astype(int))
)

In [136]:
df['sell_vol_filter'] = (
    df.groupby(['symbol', 'date'])['sell_v_pct']
    .shift(1)
    .pipe(lambda prev: (df['sell_v_pct'] > prev).astype(int))
)

In [137]:
df['candle_domination'] = np.where(
    df['ohlcv_close'] > (df['ohlcv_high'] + df['ohlcv_low']) / 2,
    'buy',
    'sell'
)

In [138]:
df['signal'] = 0

df.loc[
    (df['vol_filter'] == 1) & (df['buy_vol_filter'] == 1) & (df['domination'] == 'buy')&(df['buy_v_pct'] > 60) &(df['candle_domination'] == 'buy'),
    'signal'
] = 1

df.loc[
    (df['vol_filter'] == 1) & (df['sell_vol_filter'] == 1) & (df['domination'] == 'sell')&(df['sell_v_pct'] > 60) &(df['candle_domination'] == 'sell'),
    'signal'
] = -1

df_filtered = df[df['signal'] != 0]


In [139]:
import pandas as pd
import numpy as np

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
INPUT_CSV  = "candle_data.csv"       # change to your input file path
OUTPUT_CSV = "candle_data_with_trades_0.3.csv"
CAPITAL    = 100_000                  # 1 lakh
TARGET_PCT = 0.003                # 0.3%


# ─────────────────────────────────────────
# LOAD & SORT
# ─────────────────────────────────────────

df = df.sort_values(['symbol', 'date', 'candle_open']).reset_index(drop=True)


# ─────────────────────────────────────────
# BACKTEST
# ─────────────────────────────────────────
results = []

for idx in df[df['signal'] != 0].index:
    row    = df.loc[idx]
    entry  = row['ohlcv_close']
    signal = row['signal']
    sym    = row['symbol']

    if signal == 1:           # BUY
        sl  = row['ohlcv_low']
        tg  = round(entry * (1 + TARGET_PCT), 2)
    else:                     # SELL
        sl  = row['ohlcv_high']
        tg  = round(entry * (1 - TARGET_PCT), 2)

    qty = int(CAPITAL / entry)

    # Scan future candles of the same symbol for SL / TG
    future     = df[(df['symbol'] == sym) & (df.index > idx)]
    exit_price = None
    exit_crit  = 'OPEN'

    for _, frow in future.iterrows():
        if signal == 1:
            if frow['ohlcv_low'] <= sl:
                exit_price, exit_crit = sl,  'SL';  break
            elif frow['ohlcv_high'] >= tg:
                exit_price, exit_crit = tg,  'TG';  break
        else:
            if frow['ohlcv_high'] >= sl:
                exit_price, exit_crit = sl,  'SL';  break
            elif frow['ohlcv_low'] <= tg:
                exit_price, exit_crit = tg,  'TG';  break

    # If neither hit, exit at last available close for that symbol
    if exit_price is None:
        exit_price = df[df['symbol'] == sym].iloc[-1]['ohlcv_close']
        exit_crit  = 'OPEN'

    pnl = round((exit_price - entry) * qty, 2) if signal == 1 \
          else round((entry - exit_price) * qty, 2)

    trade_result = 'profit' if pnl > 0 else ('loss' if pnl < 0 else 'breakeven')

    results.append({
        'idx':          idx,
        'entry':        round(entry,      2),
        'exit':         round(exit_price, 2),
        'sl':           round(sl,         2),
        'tg':           tg,
        'exit_criteria': exit_crit,
        'trade_result': trade_result,
        'pnl':          pnl,
    })

res_df = pd.DataFrame(results).set_index('idx')

for col in ['entry', 'exit', 'sl', 'tg', 'exit_criteria', 'trade_result', 'pnl']:
    df[col] = res_df[col]


# ─────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────
df_fin=df[['symbol','date','candle_open','signal','entry','exit','sl','tg','exit_criteria','trade_result','pnl']]
df_fin=df_fin[df_fin['signal']!=0]
df_fin.to_csv(OUTPUT_CSV, index=False)


# ─────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────
trades = df[df['signal'] != 0]

print("=" * 45)
print("           BACKTEST SUMMARY")
print("=" * 45)
print(f"  Capital        : ₹{CAPITAL:,.0f}")
print(f"  Target         : {TARGET_PCT*100}%")
print(f"  Total trades   : {len(trades)}")
print(f"  Profit (TG hit): {(trades['trade_result'] == 'profit').sum()}")
print(f"  Loss   (SL hit): {(trades['trade_result'] == 'loss').sum()}")
print(f"  Open trades    : {(trades['exit_criteria'] == 'OPEN').sum()}")
print(f"  Net PnL        : ₹{trades['pnl'].sum():,.2f}")
print("=" * 45)
print(f"\nOutput saved to: {OUTPUT_CSV}")

           BACKTEST SUMMARY
  Capital        : ₹100,000
  Target         : 0.3%
  Total trades   : 36
  Profit (TG hit): 18
  Loss   (SL hit): 18
  Open trades    : 2
  Net PnL        : ₹1,421.03

Output saved to: candle_data_with_trades_0.3.csv


In [140]:
import pandas as pd
import numpy as np

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
INPUT_CSV  = "candle_data.csv"
OUTPUT_CSV = "candle_data_with_trades_2_4_05.csv"
CAPITAL    = 100_000                  # 1 lakh
TARGET_PCT = 0.006                    # 0.3%


# ─────────────────────────────────────────
# LOAD & SORT
# ─────────────────────────────────────────

df = df.sort_values(['symbol', 'date', 'candle_open']).reset_index(drop=True)


# ─────────────────────────────────────────
# BACKTEST
# ─────────────────────────────────────────
results        = []
open_positions = {}  # symbol -> exit index (candle where position closed)

for idx in df[df['signal'] != 0].index:
    row    = df.loc[idx]
    entry  = row['ohlcv_close']
    signal = row['signal']
    sym    = row['symbol']

    # ── SKIP if a position is still open for this symbol ──
    if sym in open_positions and idx <= open_positions[sym]:
        continue

    if signal == 1:           # BUY
        sl = row['ohlcv_low']
        tg = round(entry * (1 + TARGET_PCT), 2)
    else:                     # SELL
        sl = row['ohlcv_high']
        tg = round(entry * (1 - TARGET_PCT), 2)

    qty = int(CAPITAL / entry)

    # Scan future candles of the same symbol for SL / TG
    future     = df[(df['symbol'] == sym) & (df.index > idx)]
    exit_price = None
    exit_crit  = 'OPEN'
    exit_idx   = df[df['symbol'] == sym].index[-1]  # default: last candle

    for fidx, frow in future.iterrows():
        if signal == 1:
            if frow['ohlcv_low'] <= sl:
                exit_price, exit_crit, exit_idx = sl, 'SL', fidx
                break
            elif frow['ohlcv_high'] >= tg:
                exit_price, exit_crit, exit_idx = tg, 'TG', fidx
                break
        else:
            if frow['ohlcv_high'] >= sl:
                exit_price, exit_crit, exit_idx = sl, 'SL', fidx
                break
            elif frow['ohlcv_low'] <= tg:
                exit_price, exit_crit, exit_idx = tg, 'TG', fidx
                break

    # If neither hit, exit at last available close for that symbol
    if exit_price is None:
        exit_price = df[df['symbol'] == sym].iloc[-1]['ohlcv_close']
        exit_crit  = 'OPEN'

    # ── Mark this symbol as occupied until exit_idx ──
    open_positions[sym] = exit_idx

    pnl = round((exit_price - entry) * qty, 2) if signal == 1 \
          else round((entry - exit_price) * qty, 2)

    trade_result = 'profit' if pnl > 0 else ('loss' if pnl < 0 else 'breakeven')

    results.append({
        'idx':           idx,
        'entry':         round(entry,      2),
        'exit':          round(exit_price, 2),
        'sl':            round(sl,         2),
        'tg':            tg,
        'exit_criteria': exit_crit,
        'trade_result':  trade_result,
        'pnl':           pnl,
    })


# ─────────────────────────────────────────
# MERGE RESULTS BACK
# ─────────────────────────────────────────
res_df = pd.DataFrame(results).set_index('idx')

for col in ['entry', 'exit', 'sl', 'tg', 'exit_criteria', 'trade_result', 'pnl']:
    df[col] = res_df[col]


# ─────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────
df_fin=df[['symbol','date','candle_open','signal','entry','exit','sl','tg','exit_criteria','trade_result','pnl']]
df_fin=df_fin[df_fin['signal']!=0]
df_fin.to_csv(OUTPUT_CSV, index=False)

# ─────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────
trades = df[df['entry'].notna()]  # only rows where a trade was actually taken

print("=" * 45)
print("           BACKTEST SUMMARY")
print("=" * 45)
print(f"  Capital        : ₹{CAPITAL:,.0f}")
print(f"  Target         : {TARGET_PCT * 100}%")
print(f"  Total trades   : {len(trades)}")
print(f"  Profit (TG hit): {(trades['trade_result'] == 'profit').sum()}")
print(f"  Loss   (SL hit): {(trades['trade_result'] == 'loss').sum()}")
print(f"  Open trades    : {(trades['exit_criteria'] == 'OPEN').sum()}")
print(f"  Net PnL        : ₹{trades['pnl'].sum():,.2f}")
print("=" * 45)
print(f"\nOutput saved to: {OUTPUT_CSV}")

           BACKTEST SUMMARY
  Capital        : ₹100,000
  Target         : 0.6%
  Total trades   : 18
  Profit (TG hit): 9
  Loss   (SL hit): 9
  Open trades    : 1
  Net PnL        : ₹2,926.46

Output saved to: candle_data_with_trades_2_4_05.csv
